# `/fix` rosbag -> `fix_utm_v2.csv`

이 노트북은 rosbag2에서 `/fix` (`sensor_msgs/NavSatFix`)를 읽어 `erp42_racing_planning/resource/fix_utm_v2.csv` 형식으로 변환합니다.

가정:
- rosbag에는 `/fix` 토픽이 들어 있음
- 수집한 주행 궤적은 `center`, `left`, `right` 중 하나로 해석 가능함
- `center`로 수집했다면 좌우 lane을 `lane_separation_m / 2` 만큼 offset해서 생성함
- `left` 또는 `right`로 수집했다면 다른 쪽 lane을 전체 `lane_separation_m` 만큼 offset해서 생성함

실행 전:
- ROS 2 environment가 잡힌 Python/Jupyter에서 실행해야 함
- 예: `source /opt/ros/humble/setup.bash && source ~/workspace/ros2/erp42_racing_ws/install/setup.bash`


In [ ]:
from pathlib import Path
import csv
import math

import yaml
from pyproj import CRS, Transformer
import rosbag2_py
from rclpy.serialization import deserialize_message
from rosidl_runtime_py.utilities import get_message

# Required inputs
BAG_PATH = Path('/path/to/rosbag2')
OUTPUT_CSV = Path('/home/youngwoo/workspace/vil/erp42_racing_ws/src/erp42_racing_planning/resource/fix_utm_from_bag.csv')

# Bag / topic configuration
FIX_TOPIC = '/fix'
UTM_ZONE = 52
HEMISPHERE = 'north'  # 'north' or 'south'

# Track interpretation
SOURCE_TRACK_ROLE = 'center'  # 'center', 'left', 'right'
LANE_SEPARATION_M = 2.0

# Filtering
MIN_POINT_DISTANCE_M = 0.2

assert SOURCE_TRACK_ROLE in {'center', 'left', 'right'}
assert HEMISPHERE in {'north', 'south'}
print(f'BAG_PATH={BAG_PATH}')
print(f'OUTPUT_CSV={OUTPUT_CSV}')


In [ ]:
def detect_storage_id(bag_path: Path) -> str:
    metadata_path = bag_path / 'metadata.yaml'
    if not metadata_path.exists():
        raise FileNotFoundError(f'metadata.yaml not found: {metadata_path}')

    with metadata_path.open() as f:
        metadata = yaml.safe_load(f)

    return metadata['rosbag2_bagfile_information']['storage_identifier']


def make_transformers(zone: int, hemisphere: str):
    utm_proj4 = f'+proj=utm +zone={zone} +datum=WGS84 +units=m +no_defs'
    if hemisphere == 'south':
        utm_proj4 += ' +south'

    wgs84 = CRS.from_epsg(4326)
    utm = CRS.from_proj4(utm_proj4)
    to_utm = Transformer.from_crs(wgs84, utm, always_xy=True)
    to_wgs84 = Transformer.from_crs(utm, wgs84, always_xy=True)
    return to_utm, to_wgs84


def read_fix_messages(bag_path: Path, topic_name: str):
    storage_id = detect_storage_id(bag_path)
    reader = rosbag2_py.SequentialReader()
    storage_options = rosbag2_py.StorageOptions(uri=str(bag_path), storage_id=storage_id)
    converter_options = rosbag2_py.ConverterOptions(
        input_serialization_format='cdr',
        output_serialization_format='cdr',
    )
    reader.open(storage_options, converter_options)

    topics = {item.name: item.type for item in reader.get_all_topics_and_types()}
    if topic_name not in topics:
        raise KeyError(f'{topic_name} not found. available={list(topics)}')

    msg_type = get_message(topics[topic_name])
    messages = []
    while reader.has_next():
        topic, data, timestamp = reader.read_next()
        if topic != topic_name:
            continue

        msg = deserialize_message(data, msg_type)
        if not all(math.isfinite(v) for v in (msg.latitude, msg.longitude, msg.altitude)):
            continue

        messages.append({
            'timestamp_ns': int(timestamp),
            'lat': float(msg.latitude),
            'lon': float(msg.longitude),
            'alt': float(msg.altitude),
            'status': int(msg.status.status),
        })

    if not messages:
        raise RuntimeError(f'No valid {topic_name} messages found in {bag_path}')

    return messages


def append_utm(points, to_utm):
    converted = []
    for point in points:
        utm_x, utm_y = to_utm.transform(point['lon'], point['lat'])
        item = dict(point)
        item['utm_x'] = float(utm_x)
        item['utm_y'] = float(utm_y)
        converted.append(item)
    return converted


def filter_by_distance(points, min_distance_m: float):
    if not points:
        return []

    kept = [points[0]]
    for point in points[1:]:
        dx = point['utm_x'] - kept[-1]['utm_x']
        dy = point['utm_y'] - kept[-1]['utm_y']
        if math.hypot(dx, dy) >= min_distance_m:
            kept.append(point)
    return kept


def tangent_unit(points, index: int):
    if len(points) == 1:
        return 1.0, 0.0

    prev_index = max(index - 1, 0)
    next_index = min(index + 1, len(points) - 1)
    dx = points[next_index]['utm_x'] - points[prev_index]['utm_x']
    dy = points[next_index]['utm_y'] - points[prev_index]['utm_y']
    norm = math.hypot(dx, dy)
    if norm < 1e-9:
        return 1.0, 0.0
    return dx / norm, dy / norm


def shift_point(utm_x: float, utm_y: float, tx: float, ty: float, lateral_offset_m: float):
    left_nx = -ty
    left_ny = tx
    return utm_x + lateral_offset_m * left_nx, utm_y + lateral_offset_m * left_ny


def build_lane_rows(points, to_wgs84, source_track_role: str, lane_separation_m: float):
    rows = []
    half_sep = lane_separation_m / 2.0

    for index, point in enumerate(points):
        tx, ty = tangent_unit(points, index)

        if source_track_role == 'center':
            l1_x, l1_y = shift_point(point['utm_x'], point['utm_y'], tx, ty, half_sep)
            r1_x, r1_y = shift_point(point['utm_x'], point['utm_y'], tx, ty, -half_sep)
        elif source_track_role == 'left':
            l1_x, l1_y = point['utm_x'], point['utm_y']
            r1_x, r1_y = shift_point(point['utm_x'], point['utm_y'], tx, ty, -lane_separation_m)
        else:
            r1_x, r1_y = point['utm_x'], point['utm_y']
            l1_x, l1_y = shift_point(point['utm_x'], point['utm_y'], tx, ty, lane_separation_m)

        l1_lon, l1_lat = to_wgs84.transform(l1_x, l1_y)
        r1_lon, r1_lat = to_wgs84.transform(r1_x, r1_y)

        rows.append({
            'INDEX': index,
            'L1_UTM_X': l1_x,
            'L1_UTM_Y': l1_y,
            'L1_LON': l1_lon,
            'L1_LAT': l1_lat,
            'L1_ALT': point['alt'],
            'R1_UTM_X': r1_x,
            'R1_UTM_Y': r1_y,
            'R1_LON': r1_lon,
            'R1_LAT': r1_lat,
            'R1_ALT': point['alt'],
        })

    return rows


def write_fix_utm_csv(rows, output_csv: Path):
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = [
        'INDEX',
        'L1_UTM_X', 'L1_UTM_Y', 'L1_LON', 'L1_LAT', 'L1_ALT',
        'R1_UTM_X', 'R1_UTM_Y', 'R1_LON', 'R1_LAT', 'R1_ALT',
    ]

    with output_csv.open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


In [ ]:
to_utm, to_wgs84 = make_transformers(UTM_ZONE, HEMISPHERE)

raw_fix = read_fix_messages(BAG_PATH, FIX_TOPIC)
fix_with_utm = append_utm(raw_fix, to_utm)
filtered_fix = filter_by_distance(fix_with_utm, MIN_POINT_DISTANCE_M)
rows = build_lane_rows(filtered_fix, to_wgs84, SOURCE_TRACK_ROLE, LANE_SEPARATION_M)
write_fix_utm_csv(rows, OUTPUT_CSV)

print(f'raw_fix_count={len(raw_fix)}')
print(f'filtered_fix_count={len(filtered_fix)}')
print(f'wrote {len(rows)} rows to {OUTPUT_CSV}')
rows[:3]
